# Phase 9 — does the harness matter more than the model?

## Where this starts

| | mean | solved | note |
|---|---:|---:|---|
| classical entropy solver | 3.4431 | 246/246 | the ceiling |
| SFT + adaptive decoder @20 | **3.7642** | 242/246 | Phase 7, the baseline |
| DPO v3 + same decoder | 3.7886 | 241/246 | a clean null |
| filter only, no model | 5.1762 | 190/246 | decoder with no policy |
| SFT, no decoder | ~5.29 | — | policy with no decoder |

Read the last two rows together. The decoder without a model gets 5.18; the
model without the decoder gets ~5.29; together they get 3.76. Almost all of the
gain is in the *combination*, and the piece nobody has varied is the prompt.

## What runs here

**A. Game sweep** — 12 prompt variants × {base Qwen, SFT} × the same answers,
adaptive@20 decoder throughout. Shared answers means variant-vs-baseline is
paired, so a 0.05 difference is readable where an unpaired SE of 0.064 would
hide it.

**B. Format probe** — the same variants with *no decoder*, raw greedy
generation. Measures legal-word rate and admissible rate: what the model does
when nothing catches it.

## The twelve

| # | variant | what it tests |
|---|---|---|
| 1 | `baseline` | what every earlier phase used |
| 2 | `raw_history` | **no derived constraints — does the model deduce at all?** |
| 3 | `constraints_only` | deductions without the moves that produced them |
| 4 | `minimal` | no instructions whatsoever |
| 5 | `with_count` | 🚩 LEAKY upper bound — shows candidates remaining |
| 6 | `emoji` | 🟩🟨⬛ vs `GYB` — pure notation |
| 7 | `verbose` | feedback spelled out per letter, in prose |
| 8 | `reversed` | most recent guess first (recency vs chronology) |
| 9 | `keyboard` | the letter-status board a human actually reads |
| 10 | `few_shot` | two worked examples first |
| 11 | `guided_prose` | the same constraints as a sentence, not a table |
| 12 | `hard_mode_hint` | states the rule the decoder already enforces |

## Reading the result before it exists

- **Spread < 0.05 across variants** → prompt format is not a lever; stop here
  and go to GRPO or accept the ceiling.
- **Spread > 0.15** → format matters more than four training interventions did.
  The next SFT run should be on the winning format, and the write-up's headline
  changes.
- **`raw_history` ≈ `baseline`** → the derived block is decoration; the model is
  reading the history itself.
- **`raw_history` ≫ worse** → the harness has been doing the deduction. Say so.

Nothing here trains anything. It is measurement only.

---
# 1. Setup

In [ ]:
import os, sys, json, time, math, random, subprocess, importlib, shutil, glob
import itertools
from collections import Counter, defaultdict

def _pip(p):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
for mod, pkg in [("transformers", "transformers>=4.44"), ("peft", "peft>=0.11"),
                 ("accelerate", "accelerate>=0.30")]:
    try: importlib.import_module(mod)
    except ImportError: _pip(pkg)
import torch, transformers, peft
import torch.nn.functional as F
import numpy as np

def fix_torchao_peft_conflict():
    try: import peft.import_utils as piu
    except Exception as e: return f"unavailable ({e})"
    try: piu.is_torchao_available(); return "no conflict"
    except ImportError: pass
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                   check=False)
    importlib.invalidate_caches()
    try: piu.is_torchao_available(); return "resolved"
    except ImportError: pass
    piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as t
        t.is_torchao_available = lambda *a, **k: False
    except Exception: pass
    return "patched"
print("torchao:", fix_torchao_peft_conflict())
print("transformers", transformers.__version__, "| peft", peft.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# ============================ EDIT THIS =====================================
DATASET_DIR  = None      # sft_package dataset; None = search /kaggle/input
PREV_RUN_DIR = None      # dir holding the Phase 7 SFT adapter
# ============================================================================

MODEL_NAME  = "Qwen/Qwen2.5-0.5B-Instruct"
SFT_ADAPTER = "tree_salet_endgame"

# arm name -> adapter directory name. 'base' is the stock model and needs no
# entry. Phase 10's crossover adds a second trained arm here and runs the 2x2
# with ARMS = ["sft","sft_rawhist"], VARIANTS_TO_RUN = ["baseline","raw_history"].
ARM_ADAPTERS = {
    "sft":         SFT_ADAPTER,
    "sft_rawhist": "tree_salet_endgame_rawhist",
}

# ---- what to run ----------------------------------------------------------
RUN_GAMES   = True       # probe A
RUN_FORMAT  = True       # probe B (cheap: ~2 min for all 12) - now runs FIRST

# The `base` arm is dropped. All nine of its 2026-08-22 runs landed at
# 6.71-6.88 with 91-97% failure and a spread of 0.17: stock Qwen cannot play
# Wordle under any prompt, so the arm cannot discriminate between prompts,
# which was its whole job. It cost 84.5% of the session's wall time to answer
# nothing. The lock-in confound it was meant to break is instead settled by
# the format-crossover run (TECHNICAL.md 10C), which needs a second *trained*
# adapter, not an untrained model.
ARMS        = ["sft"]

# 246 = the full held-out set. The 2026-08-22 sweep used 100 to fit 12 variants
# x 2 arms in one session; with the base arm gone and only the variants that
# survived, the full set fits. NOTE: the 100 are a SUBSET of the 246, so this
# tightens precision on the same games - it is not an independent replication.
N_GAMES     = 246
GAME_SEED   = 20260822   # which games; fixed so runs stay comparable

# The cheap follow-up session, not the full sweep. Four variants:
#   baseline     - the control, and the gate
#   raw_history  - the load-bearing one (+0.40, and 98/100 solved: see 8b)
#   with_count   - LEAKY, but the cleanest null in the table
#   few_shot     - re-measured with the rotating non-copyable exemplars
# Set to None to run all twelve again.
VARIANTS_TO_RUN = ["baseline", "raw_history", "with_count", "few_shot"]

# ---- decoder: FIXED. This is the control, not a variable. -----------------
ADAPTIVE_THRESHOLD = 20
CONSTRAINED_CHUNK, CONSTRAINED_PRUNE = 512, True
ATTN_IMPL = "sdpa"           # eager is 5-10x slower and silently so
USE_DECISION_CACHE = True    # exact: greedy decoder + deterministic model

# ---- probe B --------------------------------------------------------------
FORMAT_PROBE_STATES = 150
FORMAT_MAX_NEW = 8

SEED = 20260822
MAX_GUESSES = 6

PHASE7_OPENER = "SALET"
PHASE7_MEAN   = 3.7642     # baseline+sft must land near this or the run is void
PHASE7 = {"sft_adaptive20": 3.7642, "control_adaptive20": 5.1762,
          "classical_entropy": 3.4431, "dpo_v3": 3.7886}

WORK_DIR     = "/kaggle/working/wordle_phase9"
RESULTS_ROOT = "/kaggle/working/results_phase9"
RESULTS_ZIP  = "/kaggle/working/wordle_phase9_results.zip"
os.makedirs(WORK_DIR, exist_ok=True); os.makedirs(RESULTS_ROOT, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s); transformers.set_seed(s)
set_seed()
print(f"\narms: {ARMS}  |  games/variant: {N_GAMES}  |  decoder: adaptive @{ADAPTIVE_THRESHOLD}")

---
# 2. Data, solver, models

Two arms. `base` is stock Qwen2.5-0.5B-Instruct with no adapter — every variant
is equally off-distribution for it, which makes it the *clean* test of whether a
prompt helps at all. `sft` is the Phase 7 adapter, which was trained on
`baseline` only, so every other variant is off-distribution for it — that
asymmetry is itself a result worth having.

In [ ]:
REQ = ["sft_package/eval/val_answers.jsonl",
       "code/wordle_solver.py", "code/generate_trajectories.py",
       "artifacts/feedback_matrix.npy"]

def _has(d):
    try: return all(os.path.exists(os.path.join(d, f)) for f in REQ)
    except OSError: return False

def find_root():
    for root in ([DATASET_DIR] if DATASET_DIR else []) + ["/kaggle/input", "."]:
        if not root or not os.path.isdir(root): continue
        if _has(root): return root
        for dp, dn, _ in os.walk(root):
            dn[:] = [d for d in dn if not d.startswith(".")]
            if _has(dp): return dp
    raise FileNotFoundError("sft_package + artifacts + code not found under /kaggle/input")

DATA_ROOT = find_root()
SFT_DIR = os.path.join(DATA_ROOT, "sft_package")
sys.path.insert(0, os.path.join(DATA_ROOT, "code"))
print("dataset:", DATA_ROOT)

from wordle_solver import (load_artifacts, SolverConfig, make_solver, play_game,
                           feedback_code, code_to_pattern, ALL_GREEN)
from generate_trajectories import derive_constraints, render_prompt

BUNDLE = load_artifacts(os.path.join(DATA_ROOT, "artifacts"), mmap=True)
VOCAB = BUNDLE.vocab
LEGAL_LOWER = [g.lower() for g in VOCAB.guesses]
LEGAL_GUESSES = set(w.upper() for w in LEGAL_LOWER)
ALL_VAL = [json.loads(l)["answer"].upper()
           for l in open(os.path.join(SFT_DIR, "eval/val_answers.jsonl"),
                         encoding="utf-8")]
assert len(LEGAL_GUESSES) == 12972 and len(ALL_VAL) == 246

# One fixed subset for every variant and every arm. Sorted after sampling so the
# order is stable regardless of how random.sample happens to enumerate.
_rng = random.Random(GAME_SEED)
ANSWERS = sorted(_rng.sample(ALL_VAL, min(N_GAMES, len(ALL_VAL))))
print(f"answers: {len(ANSWERS)} of {len(ALL_VAL)}  (seed {GAME_SEED})")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if TOKENIZER.pad_token is None: TOKENIZER.pad_token = TOKENIZER.eos_token

def base_model():
    """fp16 + sdpa, asserted. A silent fp32 load costs ~8x on a T4 and the only
    symptom is a slow run, so it is checked rather than hoped for."""
    kw = dict(trust_remote_code=True)
    if ATTN_IMPL: kw["attn_implementation"] = ATTN_IMPL
    try:
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, **kw)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                                 torch_dtype=torch.float16, **kw)
    got = next(m.parameters()).dtype
    assert got == torch.float16, f"expected fp16, got {got}"
    m.config.use_cache = False
    return m

def find_adapter(name):
    """Exact-name match only. A fallback here once loaded the wrong adapter and
    the run looked entirely normal while measuring a different model."""
    hits = []
    for root in ([PREV_RUN_DIR] if PREV_RUN_DIR else []) + [WORK_DIR, "/kaggle/input"]:
        if not root or not os.path.isdir(root): continue
        for dp, _, fs in os.walk(root):
            if "adapter_config.json" in fs and "checkpoint-" not in dp:
                hits.append(dp)
                if os.path.basename(dp.rstrip("/")) == name:
                    print(f"  adapter {name!r} -> {dp}")
                    return dp
    if hits:
        print(f"  NO adapter named {name!r}. Found instead:")
        for h in sorted(set(hits)):
            print(f"      {os.path.basename(h.rstrip(chr(47))):<24} {h}")
        print("  Refusing to fall back - that would measure the wrong model.")
    return None

# Resolve every non-'base' arm to a real adapter directory up front, so a
# missing adapter fails in cell 2 rather than an hour into the sweep.
ARM_PATHS = {}
for _arm in ARMS:
    if _arm == "base": continue
    _name = ARM_ADAPTERS.get(_arm)
    if _name is None:
        raise SystemExit(f"arm {_arm!r} has no entry in ARM_ADAPTERS.")
    _p = find_adapter(_name)
    if _p is None:
        raise SystemExit(
            f"arm {_arm!r} needs adapter {_name!r}, which is not attached. "
            f"Attach the dataset holding it, or drop the arm. Do NOT substitute "
            f"a different adapter - that voided Phase 8 v3.")
    ARM_PATHS[_arm] = _p
SFT_PATH = ARM_PATHS.get("sft")
print("arms ->", {k: os.path.basename(v) for k, v in ARM_PATHS.items()} or "base only")

---
# 3. Decoder — Phase 7, unchanged

Held fixed across every variant. If this moved, nothing below would be
attributable to the prompt.

In [ ]:
"""
constrained_decode.py — exact argmax over a fixed legal-word set.

The question this answers is

    "Which of these 12,972 legal Wordle words should I play?"

not

    "Generate arbitrary text and see whether it happens to be a word."

Nothing is filtered after the fact. The model never emits free text at all in
constrained mode: every legal word is *scored*, and the highest-scoring one is
played.

--------------------------------------------------------------------------
Definition of the score
--------------------------------------------------------------------------
For a prompt `p` and a legal word `w`, tokenize exactly as the SFT data did --
`" " + w` -- and append EOS. Then

    score(w) = log P(w | p) = sum_i log P(t_i | p, t_0..t_{i-1})

summed over the word's tokens **and the EOS token**.

Including EOS matters. Without it, a word whose token sequence is a prefix of a
longer word's is scored on a strictly smaller set of constraints and is
systematically over-ranked; with EOS the scores are log-probabilities of
complete strings, so they are directly comparable across different token
lengths. No length normalisation is applied: `score(w)` is exactly the
probability the model assigns to playing `w`, which is the quantity we want to
argmax. (`length_normalise=True` is available for a sensitivity check but is a
heuristic, not the default.)

--------------------------------------------------------------------------
How it is computed
--------------------------------------------------------------------------
Naively this is 12,972 forward passes per turn. Instead:

1. The prompt is run **once** with `use_cache=True`, giving a KV cache and the
   next-token distribution `lp0` over the whole vocabulary.
2. `lp0` already gives the first-token log-probability of every legal word, for
   free -- no forward pass.
3. The remaining tokens are scored by teacher forcing: the prompt's KV cache is
   expanded to a batch of `chunk` rows and the padded `[chunk, L]` word-token
   matrix is pushed through in one forward pass. Causal masking makes the
   right-hand padding inert.

Exact branch-and-bound pruning (`prune=True`, on by default and *exact*):
every per-token log-probability is <= 0, so

    score(w) <= lp0[first_token(w)]

is a valid upper bound. Words are visited in descending order of that bound;
once the best fully-scored word beats the bound of every unvisited word, no
unvisited word can win and the scan stops. The returned argmax is identical to
scoring all 12,972 -- `verify_against_full()` asserts exactly that.

--------------------------------------------------------------------------
What is NOT done here
--------------------------------------------------------------------------
- The candidate-answer list is never consulted. The scorer ranks the full legal
  guess pool; it has no idea which words are still possible.
- The hidden answer is never consulted.
- Repeats are not banned by default (`banned` is opt-in), because banning them
  would be a policy change, not a vocabulary constraint.
"""

import numpy as np
import torch
import torch.nn.functional as F

NEG_INF = float("-inf")


# ---------------------------------------------------------------------------
# KV-cache compatibility
#
# transformers has moved the cache representation twice (legacy tuple ->
# DynamicCache.key_cache/value_cache -> Cache.layers). Rather than pin a
# version, read whichever layout is present and rebuild through the same one.
# `LegalWordScorer.self_test()` verifies the result numerically at runtime, so
# a layout this shim gets wrong fails loudly instead of scoring garbage.
# ---------------------------------------------------------------------------
def _cache_layers(cache):
    if hasattr(cache, "layers"):                        # transformers >= 5
        return [(lyr.keys, lyr.values) for lyr in cache.layers]
    if hasattr(cache, "key_cache"):                     # transformers 4.x
        return list(zip(cache.key_cache, cache.value_cache))
    return [(k, v) for k, v in cache]                   # legacy tuple-of-tuples


def _rebuild_cache(template, layers):
    """Rebuild a cache object of the same kind as `template` from `layers`."""
    if hasattr(template, "layers"):
        import copy
        new = copy.deepcopy(template)
        for lyr, (k, v) in zip(new.layers, layers):
            lyr.keys, lyr.values = k, v
        return new
    if hasattr(template, "key_cache"):
        from transformers.cache_utils import DynamicCache
        new = DynamicCache()
        new.key_cache = [k for k, _ in layers]
        new.value_cache = [v for _, v in layers]
        return new
    return tuple(layers)


def expand_cache(cache, n):
    """Repeat a batch-1 KV cache to batch `n` without recomputing the prompt."""
    out = []
    for k, v in _cache_layers(cache):
        assert k.shape[0] == 1, f"expected batch-1 cache, got {k.shape[0]}"
        out.append((k.expand(n, *k.shape[1:]).contiguous(),
                    v.expand(n, *v.shape[1:]).contiguous()))
    return _rebuild_cache(cache, out)


# ---------------------------------------------------------------------------
class LegalWordScorer:
    """Scores every word in a fixed legal set under a causal LM."""

    def __init__(self, tokenizer, words, device="cuda", chunk=512,
                 length_normalise=False):
        self.tok = tokenizer
        self.words = list(words)
        self.device = device
        self.chunk = chunk
        self.length_normalise = length_normalise
        self.n = len(self.words)

        eos = tokenizer.eos_token_id
        seqs = []
        for w in self.words:
            # EXACTLY the training-time tokenization: " " + WORD, then EOS.
            ids = tokenizer(" " + w, add_special_tokens=False)["input_ids"]
            seqs.append(ids + [eos])
        self.max_len = max(len(s) for s in seqs)

        pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos
        tokens = np.full((self.n, self.max_len), pad, dtype=np.int64)
        mask = np.zeros((self.n, self.max_len), dtype=np.float32)
        for i, s in enumerate(seqs):
            tokens[i, :len(s)] = s
            mask[i, :len(s)] = 1.0

        self.tokens = torch.from_numpy(tokens).to(device)      # [N, L]
        self.mask = torch.from_numpy(mask).to(device)          # [N, L]
        self.lengths = torch.from_numpy(mask.sum(1)).to(device)
        self.first_tok = self.tokens[:, 0].clone()             # [N]
        self.index = {w: i for i, w in enumerate(self.words)}

    # -- internals ----------------------------------------------------------
    @torch.no_grad()
    def _prompt_pass(self, model, prompt):
        ids = self.tok(prompt, return_tensors="pt",
                       add_special_tokens=False)["input_ids"].to(self.device)
        out = model(input_ids=ids, use_cache=True)
        lp0 = F.log_softmax(out.logits[0, -1].float(), dim=-1)   # [V]
        return out.past_key_values, lp0, ids.shape[1]

    @torch.no_grad()
    def _score_rows(self, model, cache, lp0, prompt_len, rows):
        """Exact log P(w | prompt) for the word indices in `rows`."""
        idx = rows.to(self.device)
        toks = self.tokens[idx]                                  # [C, L]
        msk = self.mask[idx]
        C, L = toks.shape

        total = lp0[toks[:, 0]] * msk[:, 0]                      # token 0, free
        if L > 1:
            big = expand_cache(cache, C)
            attn = torch.ones(C, prompt_len + L, dtype=torch.long,
                              device=self.device)
            pos = torch.arange(prompt_len, prompt_len + L,
                               device=self.device).unsqueeze(0).expand(C, L)
            out = model(input_ids=toks, past_key_values=big,
                        attention_mask=attn, position_ids=pos, use_cache=False)
            # logits[:, i] predicts token i+1, so positions 1..L-1 read 0..L-2.
            for i in range(1, L):
                lp = F.log_softmax(out.logits[:, i - 1].float(), dim=-1)
                total = total + lp.gather(1, toks[:, i:i + 1]).squeeze(1) * msk[:, i]
            del out, big
        if self.length_normalise:
            total = total / msk.sum(1)
        return total

    # -- public API ---------------------------------------------------------
    @torch.no_grad()
    def score_all(self, model, prompt):
        """Score every legal word. No pruning. Returns a [N] float tensor."""
        cache, lp0, plen = self._prompt_pass(model, prompt)
        out = torch.empty(self.n, dtype=torch.float32, device=self.device)
        for s in range(0, self.n, self.chunk):
            rows = torch.arange(s, min(s + self.chunk, self.n))
            out[rows.to(self.device)] = self._score_rows(
                model, cache, lp0, plen, rows)
        return out

    @torch.no_grad()
    def argmax(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Highest-scoring legal word.

        With `prune=True` this is the *same* word `score_all().argmax()` gives;
        the bound is exact, not a heuristic. Returns
        `(word, score, n_chunks_scored, margin)`.

        `margin` is the gap to the runner-up **among words actually scored**.
        With pruning on, unvisited words are known to score below the winner but
        could sit above the runner-up, so `margin` is an upper bound on the true
        margin. It is a diagnostic, never an input to a decision.
        """
        cache, lp0, plen = self._prompt_pass(model, prompt)

        bound = lp0[self.first_tok].clone()                      # [N] upper bd
        # `ban_mask` marks words that must NOT be returned, for any reason.
        # It is applied to the bound (so they sort last and prune early) AND to
        # the scores (so one cannot win from the tail of a live chunk). Masking
        # only the bound is a real bug we shipped once: the excluded word still
        # received a genuine score and could come out on top.
        ban_mask = torch.zeros(self.n, dtype=torch.bool, device=self.device)

        if allowed_idx is not None:
            keep = torch.zeros(self.n, dtype=torch.bool, device=self.device)
            keep[torch.as_tensor(np.asarray(allowed_idx), device=self.device)] = True
            ban_mask |= ~keep
            bound = bound.masked_fill(~keep, NEG_INF)

        if banned:
            hit = [self.index[b] for b in banned if b in self.index]
            if hit:
                ix = torch.tensor(hit, device=self.device)
                bound[ix] = NEG_INF     # sorts them last
                ban_mask[ix] = True     # and removes them from the scores

        # Only ever visit words that could win. Sorting the whole vocabulary and
        # relying on the -inf bound to skip the rest still pushes a full chunk
        # of masked-out words through the model: a 3-word admissible set cost
        # MORE than an unfiltered decision (measured 87s vs 32s on CPU) because
        # both scored 512 rows. Restricting the scan pool to the admissible set
        # makes a small set genuinely cheap, which is the common case once the
        # feedback filter bites.
        if allowed_idx is not None:
            pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
            pool = pool[~ban_mask[pool]]
            if pool.numel() == 0:                    # everything banned
                pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
        else:
            pool = torch.arange(self.n, device=self.device)
        order = pool[torch.argsort(bound[pool], descending=True)]
        n_scan = int(order.numel())

        best_i, best_s, second = -1, NEG_INF, NEG_INF
        n_chunks = 0
        for s in range(0, n_scan, self.chunk):
            rows = order[s:s + self.chunk]
            if bound[rows[0]] == NEG_INF:
                break                                    # all remaining banned
            if best_s >= bound[rows[0]].item():
                break                # no unvisited word can beat the incumbent
            sc = self._score_rows(model, cache, lp0, plen, rows.cpu())
            # A banned word can still land in the tail of an otherwise-live
            # chunk. Masking the bound alone is not enough -- the score has to
            # be masked too, or the ban is silently ignored.
            sc = sc.masked_fill(ban_mask[rows], NEG_INF)
            n_chunks += 1
            top2 = torch.topk(sc, min(2, sc.numel()))
            if top2.values[0].item() > best_s:
                second = max(second, best_s)
                if top2.values.numel() > 1:
                    second = max(second, top2.values[1].item())
                best_s = top2.values[0].item()
                best_i = rows[top2.indices[0]].item()
            elif top2.values[0].item() > second:
                second = top2.values[0].item()

        margin = None if second == NEG_INF else best_s - second
        return self.words[best_i], best_s, n_chunks, margin

    @torch.no_grad()
    def rank_of(self, scores, words):
        """1-based ranks of `words` under a full `score_all` vector."""
        order = torch.argsort(scores, descending=True)
        pos = torch.empty_like(order)
        pos[order] = torch.arange(order.numel(), device=order.device)
        return {w: int(pos[self.index[w]].item()) + 1
                for w in words if w in self.index}

    # -- feedback-consistent selection (Phase 7) ----------------------------
    @torch.no_grad()
    def select(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Pick a word, optionally restricted to an admissible subset.

        `allowed_idx` is an index array into `self.words`. It is intended to
        carry the FEEDBACK-CONSISTENT set: the legal words that would have
        produced exactly the feedback already observed. That set is a pure
        function of the prompt (history + the public word list) -- it never
        touches the answer list -- so restricting to it is the same class of
        move as restricting to legal words.

        Returns a dict, not a tuple, so callers can record why a word was
        chosen. The distinction matters for interpreting results:

            forced       len(allowed) == 1. The filter determined the word.
                         The model contributed nothing and this must NOT be
                         counted as a model decision.
            model_chosen len(allowed) > 1. The model ranked the admissible
                         words and picked one.

        Reporting these together would let a decoder improvement masquerade as
        a model improvement.
        """
        n_allowed = self.n if allowed_idx is None else int(len(allowed_idx))

        if allowed_idx is not None and n_allowed == 0:
            # Cannot happen while the answer is legal and the feedback honest,
            # but degrade to the full pool rather than crash a 4-hour run.
            allowed_idx, n_allowed = None, self.n

        if allowed_idx is not None and n_allowed == 1:
            i = int(allowed_idx[0])
            return {"word": self.words[i], "score": None, "n_chunks": 0,
                    "margin": None, "n_allowed": 1, "forced": True,
                    "model_chosen": False}

        w, s, nch, margin = self.argmax(model, prompt, banned=banned,
                                        allowed_idx=allowed_idx, prune=prune)
        return {"word": w, "score": s, "n_chunks": nch, "margin": margin,
                "n_allowed": n_allowed, "forced": False, "model_chosen": True}

    # -- correctness --------------------------------------------------------
    @torch.no_grad()
    def self_test(self, model, prompt, n_probe=32, atol=None):
        """Check cache-reuse scoring against naive full-sequence scoring.

        This is the guard on the KV-cache shim above. A shim that mishandles a
        transformers version produces essentially random scores -- wrong by
        many nats, with the ordering destroyed. That is the failure this must
        catch, and it is enormous.

        What it must NOT flag is float16 rounding. The two paths run different
        matmul shapes (one sequence of length P+L, versus a batch of L-token
        rows against an expanded P-token cache), so fp16 reduction order
        differs and summed log-probs disagree at the 0.01-0.1 nat level. That
        is arithmetic noise, not a broken cache.

        So the test is two-sided:
          * absolute deviation under a dtype-aware tolerance, and
          * the two score vectors still rank the probe words the same way
            (correlation ~1). A broken shim cannot preserve the ranking.

        Note this discrepancy is a *validation* artifact only. Every one of the
        12,972 words is scored through the same fast path, so the ranking the
        argmax is read off is internally consistent.

        Probe indices are spread evenly across the whole word list, not taken
        from the front: if the prompt cache were mutated in place by the first
        chunk's forward pass, only words in *later* chunks would be wrong, and
        a probe drawn from the front would miss it entirely.
        """
        dtype = next(model.parameters()).dtype
        if atol is None:
            atol = 0.02 if dtype in (torch.float32, torch.float64) else 0.40

        fast = self.score_all(model, prompt)
        p_ids = self.tok(prompt, return_tensors="pt",
                         add_special_tokens=False)["input_ids"].to(self.device)
        n_probe = min(n_probe, self.n)
        probe = [int(round(i * (self.n - 1) / max(n_probe - 1, 1)))
                 for i in range(n_probe)]

        got, ref_all = [], []
        for i in probe:
            ids = self.tokens[i][self.mask[i] > 0].unsqueeze(0)
            full = torch.cat([p_ids, ids], dim=1)
            logits = model(input_ids=full, use_cache=False).logits[0].float()
            lp = F.log_softmax(logits, dim=-1)
            start = p_ids.shape[1] - 1
            ref = sum(lp[start + j, ids[0, j]].item() for j in range(ids.shape[1]))
            if self.length_normalise:
                ref /= ids.shape[1]
            ref_all.append(ref)
            got.append(fast[i].item())

        a = np.asarray(got, dtype=np.float64)
        b = np.asarray(ref_all, dtype=np.float64)
        worst = float(np.max(np.abs(a - b)))
        spread = float(b.max() - b.min())
        corr = (float(np.corrcoef(a, b)[0, 1])
                if a.std() > 1e-9 and b.std() > 1e-9 else 1.0)

        assert corr > 0.999, (
            f"constrained scorer does not preserve the ranking of the naive "
            f"scorer (corr={corr:.6f}). The KV-cache shim in expand_cache() "
            f"does not match this transformers version -- do not trust "
            f"constrained results.")
        assert worst < atol, (
            f"constrained scorer disagrees with naive scoring by {worst:.4f} "
            f"nats (tol {atol} for dtype {dtype}), across a probe score spread "
            f"of {spread:.1f} nats. Ranking is preserved (corr={corr:.6f}), so "
            f"this looks like arithmetic noise rather than a broken cache -- "
            f"but it is larger than expected. Investigate before trusting the "
            f"numbers.")
        return {"max_abs_dev": worst, "corr": corr, "spread": spread,
                "atol": atol, "dtype": str(dtype), "n_probe": n_probe}

    @torch.no_grad()
    def verify_against_full(self, model, prompt):
        """Assert the pruned argmax equals the unpruned argmax."""
        full = self.score_all(model, prompt)
        w_full = self.words[int(full.argmax().item())]
        w_prune, _, n_chunks, _ = self.argmax(model, prompt, prune=True)
        assert w_full == w_prune, (
            f"pruning changed the answer: full={w_full} pruned={w_prune}. "
            f"The branch-and-bound bound is wrong.")
        return w_full, n_chunks


# ---------------------------------------------------------------------------
class HardModeFilter:
    """The legal words consistent with every piece of feedback received.

    A word `w` is admissible iff, for every past guess `g` with observed
    pattern `p`, `feedback_code(g, w) == p`. That is precisely the set a
    hard-mode Wordle player can compute from their own board.

    WHAT THIS IS NOT: the candidate set. Two sets are easy to conflate and the
    difference is the whole justification --

        candidate set   answers consistent with feedback   pool = 2,315 answers
                        -> uses the ANSWER LIST, privileged, never used here
        hard-mode set   legal guesses consistent with it   pool = 12,972 legal
                        -> a pure function of the prompt + the public word list

    The model is never shown this set, its size, the answer, or the answer
    list. It is a decoder-side restriction on which words may be selected,
    exactly like the legal-word constraint.

    Refinement is incremental, so cost is dominated by the first turn and
    collapses immediately after (measured: 12,972 -> ~211 -> ~7 -> ~2).
    """

    def __init__(self, legal_words_lower, feedback_code_fn):
        self.words = list(legal_words_lower)
        self._fb = feedback_code_fn
        self.history = []

    def refine(self, guess, code):
        """Apply one (guess, feedback) pair. `guess` lower-case, `code` int."""
        g = guess.lower()
        self.words = [w for w in self.words if self._fb(g, w) == code]
        self.history.append((g, code))
        return self

    def indices(self, scorer):
        """Index array into `scorer.words` (which are upper-case)."""
        return np.array([scorer.index[w.upper()] for w in self.words
                         if w.upper() in scorer.index], dtype=np.int64)

    def contains(self, word):
        return word.lower() in self.words

    def __len__(self):
        return len(self.words)


LEGAL_WORDS_SORTED = sorted(LEGAL_GUESSES)
SCORER = None
def build_scorer():
    global SCORER
    if SCORER is None:
        SCORER = LegalWordScorer(TOKENIZER, LEGAL_WORDS_SORTED, device="cuda",
                                 chunk=CONSTRAINED_CHUNK)
    return SCORER
print("decoder ready")

---
# 4. The twelve prompt variants

Every variant is `f(turn, history, max_guesses) -> str` ending in `Next guess:`,
so the decoder scores continuations identically across all of them. Nothing here
sees the answer, the answer list, or the candidate set — except `with_count`,
which is marked leaky and reported separately.

The cell prints all twelve rendered on the same state. **Read that output**
before letting the sweep run for an hour; a variant that renders wrongly will
still produce a plausible-looking number.

In [ ]:
"""
prompt_variants.py - twelve ways of showing the same Wordle state.

WHY THIS EXISTS

The largest measured effect in this project is not the model. It is the
scaffolding: a feedback-consistency constraint on the decoder was worth ~1.5
guesses, while eight phases of training were worth a fraction of that. This
module attacks that finding directly by varying the OTHER half of the
scaffolding - the prompt - with the decoder held fixed.

The variant that matters most is `raw_history`. The baseline prompt hands the
model `Confirmed letters : p1=A`, computed by the classical solver. If
performance collapses without it, a large part of what we have been calling
"the model playing Wordle" is the harness playing Wordle.

CONTRACT

Every variant is `f(turn, history, max_guesses) -> str` and must end with
"Next guess:" so the constrained decoder can score continuations identically
across variants. `history` is a list of `(guess_lower, pattern)` where pattern
is five characters from GYB.

LEAKAGE

One variant, `with_count`, deliberately shows the surviving-candidate count.
It is a solver-side quantity a player cannot see and it is marked `leaky=True`.
It exists only as an upper bound - "how much would the hint alone buy?" - and
its number must never be quoted as a fair result.
"""

# (sys.path bootstrap and imports supplied by the cells above)

RULES = (
    "You are playing Wordle. Deduce the hidden 5-letter word.\n"
    "After each guess you receive 5 feedback characters:\n"
    "  G = correct letter in the correct position\n"
    "  Y = letter is in the word but in a different position\n"
    "  B = letter is not in the word (or all its copies are already "
    "accounted for)\n"
)

TILE = {"G": "\U0001F7E9", "Y": "\U0001F7E8", "B": "⬛"}
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"


# --------------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------------
def _hist_lines(history, upper=True):
    out = []
    for i, (g, p) in enumerate(history, 1):
        w = g.upper() if upper else g
        out.append(f"  {i}. {w} -> {p}")
    return out


def _deduced_block(history):
    """The structured constraint block used by the baseline."""
    c = derive_constraints(history)
    lines = ["Deduced so far:"]
    greens = " ".join(f"p{i+1}={ch.upper()}"
                      for i, ch in enumerate(c["green_pattern"])
                      if ch and ch not in "._ ")
    lines.append(f"  Confirmed letters : {greens or '(none)'}")
    pres = ", ".join(sorted(x.upper() for x in c["present_letters"]))
    lines.append(f"  Letters present   : {pres or '(none)'}")
    if c["exact_counts"]:
        ex = ", ".join(f"{k.upper()}x{v}" for k, v in sorted(c["exact_counts"].items()))
        lines.append(f"  Exact counts      : {ex}")
    absent = ", ".join(sorted(x.upper() for x in c["absent_letters"]))
    lines.append(f"  Letters absent    : {absent or '(none)'}")
    fp = {k: sorted(v) for k, v in c["forbidden_positions"].items() if v}
    if fp:
        rl = ", ".join(f"{k.upper()} not at {v}" for k, v in sorted(fp.items()))
        lines.append(f"  Ruled-out spots   : {rl}")
    return "\n".join(lines)


def _turn_line(turn, max_guesses):
    return f"Guess {turn} of {max_guesses} ({max_guesses - turn + 1} remaining)"


def _empty(history):
    return not history


# --------------------------------------------------------------------------
# 1. baseline — exactly what every earlier phase used
# --------------------------------------------------------------------------
def v_baseline(turn, history, max_guesses):
    return render_prompt(
        turn=turn, history=list(history),
        constraints=derive_constraints(history),
        n_candidates=0, guesses_remaining=max_guesses - turn + 1,
        max_guesses=max_guesses, candidates=None, show_candidate_count=False)


# --------------------------------------------------------------------------
# 2. raw_history — no derived constraints. THE key variant.
# --------------------------------------------------------------------------
def v_raw_history(turn, history, max_guesses):
    h = ("History: (none - this is the opening guess)" if _empty(history)
         else "History:\n" + "\n".join(_hist_lines(history)))
    return f"{RULES}\n\n{h}\n\n{_turn_line(turn, max_guesses)}\n\nNext guess:"


# --------------------------------------------------------------------------
# 3. constraints_only — the deductions, but not the moves that produced them
# --------------------------------------------------------------------------
def v_constraints_only(turn, history, max_guesses):
    d = ("Nothing is known yet." if _empty(history) else _deduced_block(history))
    return f"{RULES}\n\n{d}\n\n{_turn_line(turn, max_guesses)}\n\nNext guess:"


# --------------------------------------------------------------------------
# 4. minimal — no instructions at all
# --------------------------------------------------------------------------
def v_minimal(turn, history, max_guesses):
    h = "\n".join(f"{g.upper()} {p}" for g, p in history)
    return (h + "\n\nNext guess:") if h else "Next guess:"


# --------------------------------------------------------------------------
# 5. with_count — LEAKY. Upper bound only.
# --------------------------------------------------------------------------
def v_with_count(turn, history, max_guesses, n_candidates=None):
    base = v_baseline(turn, history, max_guesses)
    if n_candidates is None:
        return base
    ins = f"Possible answers remaining: {n_candidates}\n\n"
    return base.replace("Next guess:", ins + "Next guess:")


# --------------------------------------------------------------------------
# 6. emoji — same information, different notation
# --------------------------------------------------------------------------
def v_emoji(turn, history, max_guesses):
    rules = (
        "You are playing Wordle. Deduce the hidden 5-letter word.\n"
        "After each guess you receive 5 tiles:\n"
        f"  {TILE['G']} = correct letter in the correct position\n"
        f"  {TILE['Y']} = letter is in the word but in a different position\n"
        f"  {TILE['B']} = letter is not in the word\n")
    lines = [f"  {i}. {g.upper()} {''.join(TILE[c] for c in p)}"
             for i, (g, p) in enumerate(history, 1)]
    h = ("History: (none - this is the opening guess)" if not lines
         else "History:\n" + "\n".join(lines))
    return f"{rules}\n\n{h}\n\n{_turn_line(turn, max_guesses)}\n\nNext guess:"


# --------------------------------------------------------------------------
# 7. verbose — feedback spelled out per letter, in prose
# --------------------------------------------------------------------------
def v_verbose(turn, history, max_guesses):
    if _empty(history):
        body = "You have not guessed yet."
    else:
        parts = []
        for i, (g, p) in enumerate(history, 1):
            bits = []
            for j, (ch, t) in enumerate(zip(g.upper(), p)):
                if t == "G":
                    bits.append(f"{ch} is correct at position {j+1}")
                elif t == "Y":
                    bits.append(f"{ch} is in the word but not at position {j+1}")
                else:
                    bits.append(f"{ch} is not in the word")
            parts.append(f"Guess {i} was {g.upper()}: " + "; ".join(bits) + ".")
        body = "\n".join(parts)
    return (f"{RULES}\n\nWhat you have learned:\n{body}\n\n"
            f"{_turn_line(turn, max_guesses)}\n\nNext guess:")


# --------------------------------------------------------------------------
# 8. reversed — most recent guess first
# --------------------------------------------------------------------------
def v_reversed(turn, history, max_guesses):
    rev = list(reversed(history))
    lines = [f"  {g.upper()} -> {p}" for g, p in rev]
    h = ("History: (none - this is the opening guess)" if not lines
         else "History (most recent first):\n" + "\n".join(lines))
    return (f"{RULES}\n\n{h}\n\n{_deduced_block(history) if history else ''}\n\n"
            f"{_turn_line(turn, max_guesses)}\n\nNext guess:")


# --------------------------------------------------------------------------
# 9. keyboard — the on-screen letter status board a human actually looks at
# --------------------------------------------------------------------------
def v_keyboard(turn, history, max_guesses):
    status = {}
    for g, p in history:
        for ch, t in zip(g.upper(), p):
            rank = {"B": 0, "Y": 1, "G": 2}
            if ch not in status or rank[t] > rank[status[ch]]:
                status[ch] = t
    rows = []
    for group in ("QWERTYUIOP", "ASDFGHJKL", "ZXCVBNM"):
        rows.append("  " + " ".join(
            f"{c}{'*' if status.get(c) == 'G' else '+' if status.get(c) == 'Y' else '-' if status.get(c) == 'B' else ' '}"
            for c in group))
    kb = ("Letter status  (* = right spot, + = in word, - = not in word):\n"
          + "\n".join(rows))
    h = ("History: (none - this is the opening guess)" if _empty(history)
         else "History:\n" + "\n".join(_hist_lines(history)))
    return (f"{RULES}\n\n{h}\n\n{kb}\n\n"
            f"{_turn_line(turn, max_guesses)}\n\nNext guess:")


# --------------------------------------------------------------------------
# 10. few_shot — two worked examples before the real state
# --------------------------------------------------------------------------
# The exemplars are COMPUTED, not hand-written. The first draft of this block
# was typed by hand and listed B as an absent letter in a game where B appeared
# in neither guess - a worked example teaching the model a false deduction. The
# patterns and constraint blocks below are now derived by the same functions the
# real prompts use, so an exemplar cannot disagree with the game it illustrates.
# A FIXED exemplar list is the defect that voided the first run of this
# variant. Two exemplars ending "Next guess: CRAFT" / "Next guess: BOBBY" made
# the final word a constant attractor: the model opened BOOBY on all 100 games
# and the 6.06 mean measured verbatim copying, not few-shot learning.
#
# Nothing can structurally stop a model copying the last example, so this does
# two things instead. The pool is rotated per state, so a copier emits eight
# different words rather than one - it can no longer look like a stable policy.
# And the harness measures the copy rate directly (`shot_copy_pct`), so the
# variant is declared valid or invalid on a number rather than on inspection.
#
# Each entry is (answer, guesses_so_far, next_guess); `next_guess` is asserted
# legal and consistent with its own feedback at build time.
_SHOT_POOL = [
    ("chart", ["salet"],           "audit"),   # 20 admissible - wide probe
    ("hobby", ["salet", "crony"],  "howdy"),   #  7
    ("plumb", ["salet", "chord"],  "imply"),   # 17
    ("wince", ["salet", "dingo"],  "mince"),   #  2 - near the end
    ("proxy", ["salet", "chirp"],  "proud"),   #  4
    ("thumb", ["salet", "conic"],  "truth"),   #  6
    ("berth", ["salet", "brine"],  "berth"),   #  1 - forced; teaches closing
    ("glaze", ["salet"],           "blame"),   # 49
]

_N_SHOTS = 2          # exemplars per prompt; the pool is what rotates

# Kept for callers that still import the old name.
_SHOT_GAMES = _SHOT_POOL


def _render_shot(n, answer, guesses, nxt):
    from wordle_solver import feedback_code, code_to_pattern
    hist = [(g, code_to_pattern(feedback_code(g, answer))) for g in guesses]
    return "\n".join(
        [f"Example {n}", "History:"]
        + _hist_lines(hist)
        + [_deduced_block(hist), f"Next guess: {nxt.upper()}", ""])


def _shot_index(history):
    """Deterministic rotation offset. Hash of the live state, so the same state
    always renders the same exemplars (the run stays reproducible) while
    different states see different ones."""
    import hashlib
    h = hashlib.md5(repr(tuple(history)).encode("utf-8")).hexdigest()
    return int(h[:8], 16)


def shot_words(history):
    """The exemplar next-guesses visible in this prompt. The harness uses this
    to compute the copy rate."""
    k, n = _shot_index(history), len(_SHOT_POOL)
    return [_SHOT_POOL[(k + i) % n][2].upper() for i in range(_N_SHOTS)]


def _shots(history):
    k, n = _shot_index(history), len(_SHOT_POOL)
    picks = [_SHOT_POOL[(k + i) % n] for i in range(_N_SHOTS)]
    return "\n".join(_render_shot(i, *p) for i, p in enumerate(picks, 1)) + "\n"


def v_few_shot(turn, history, max_guesses):
    return RULES + "\n\n" + _shots(history) + "Now your game.\n" + \
        v_baseline(turn, history, max_guesses).replace(RULES, "").lstrip("\n")


# --------------------------------------------------------------------------
# 11. guided_prose — the same deductions written as a sentence
# --------------------------------------------------------------------------
def v_guided_prose(turn, history, max_guesses):
    if _empty(history):
        body = "You know nothing yet, so pick a word that tests common letters."
    else:
        c = derive_constraints(history)
        g = [f"position {i+1} is {ch.upper()}"
             for i, ch in enumerate(c["green_pattern"])
             if ch and ch not in "._ "]
        pres = sorted(x.upper() for x in c["present_letters"])
        ab = sorted(x.upper() for x in c["absent_letters"])
        s = []
        if g:
            s.append("You know " + ", ".join(g) + ".")
        if pres:
            s.append("The word contains " + ", ".join(pres) + ".")
        if ab:
            s.append("It does not contain " + ", ".join(ab) + ".")
        fp = {k: sorted(v) for k, v in c["forbidden_positions"].items() if v}
        if fp:
            s.append("Also, " + "; ".join(
                f"{k.upper()} is not at position {[i+1 for i in v]}"
                for k, v in sorted(fp.items())) + ".")
        body = " ".join(s)
    h = "" if _empty(history) else "History:\n" + "\n".join(_hist_lines(history)) + "\n\n"
    return (f"{RULES}\n\n{h}{body}\n\n"
            f"{_turn_line(turn, max_guesses)}\n\nNext guess:")


# --------------------------------------------------------------------------
# 12. hard_mode_hint — tell it the rule the decoder already enforces
# --------------------------------------------------------------------------
def v_hard_mode_hint(turn, history, max_guesses):
    base = v_baseline(turn, history, max_guesses)
    hint = ("Play only words that are consistent with every clue above.\n\n")
    return base.replace("Next guess:", hint + "Next guess:")


# --------------------------------------------------------------------------
VARIANTS = {
    "baseline":         dict(fn=v_baseline,        leaky=False,
                             note="what every earlier phase used"),
    "raw_history":      dict(fn=v_raw_history,     leaky=False,
                             note="no derived constraints - does the model deduce?"),
    "constraints_only": dict(fn=v_constraints_only, leaky=False,
                             note="deductions without the moves"),
    "minimal":          dict(fn=v_minimal,         leaky=False,
                             note="no instructions at all"),
    "with_count":       dict(fn=v_with_count,      leaky=True,
                             note="LEAKY upper bound - shows candidates remaining"),
    "emoji":            dict(fn=v_emoji,           leaky=False,
                             note="tiles instead of GYB"),
    "verbose":          dict(fn=v_verbose,         leaky=False,
                             note="feedback spelled out per letter"),
    "reversed":         dict(fn=v_reversed,        leaky=False,
                             note="most recent guess first"),
    "keyboard":         dict(fn=v_keyboard,        leaky=False,
                             note="letter status board, as a human sees"),
    "few_shot":         dict(fn=v_few_shot,        leaky=False,
                             note="two worked examples first"),
    "guided_prose":     dict(fn=v_guided_prose,    leaky=False,
                             note="constraints as a sentence"),
    "hard_mode_hint":   dict(fn=v_hard_mode_hint,  leaky=False,
                             note="states the rule the decoder enforces"),
}


def render(name, turn, history, max_guesses, n_candidates=None):
    spec = VARIANTS[name]
    if name == "with_count":
        return spec["fn"](turn, history, max_guesses, n_candidates)
    return spec["fn"](turn, history, max_guesses)


# The few-shot exemplars are computed, but "computed" only means self-consistent
# - it does not prove the words are playable. A worked example using an illegal
# word teaches the model to emit illegal words, and the decoder would hide it.
for _ans, _gs, _nxt in _SHOT_POOL:
    assert _nxt.upper() in LEGAL_GUESSES, f"exemplar next-guess {_nxt!r} is not legal"
    for _g in _gs:
        assert _g.upper() in LEGAL_GUESSES, f"exemplar guess {_g!r} is not legal"
    assert all(feedback_code(_g, _nxt) == feedback_code(_g, _ans) for _g in _gs),         f"exemplar next-guess {_nxt!r} contradicts its own feedback"

# The defect that voided the first run: two fixed exemplars made the last
# Next-guess a constant attractor and the model opened BOOBY on every game.
# Distinct next-guesses plus per-state rotation mean a copier can no longer
# emit one stable word; play() measures how often it copies at all.
_nx = [n.upper() for _, _, n in _SHOT_POOL]
assert len(set(_nx)) == len(_nx), "exemplar next-guesses must be distinct"
assert len(_SHOT_POOL) >= 4, "too few exemplars to rotate meaningfully"
print(f"few-shot: {len(_SHOT_POOL)} exemplars, legal and self-consistent; "
      f"{_N_SHOTS} shown per prompt, rotated by state")

NAMES = list(VARIANTS) if VARIANTS_TO_RUN is None else list(VARIANTS_TO_RUN)
for n in NAMES:
    assert n in VARIANTS, f"unknown variant {n!r}"

_demo = [("salet", "BYBBB"), ("crony", "BYBBG")]
print("=" * 78)
print(f"{len(NAMES)} variants, rendered on turn 3 after SALET->BYBBB, CRONY->BYBBG")
print("=" * 78)
for n in NAMES:
    p = render(n, 3, _demo, MAX_GUESSES, n_candidates=17)
    tag = "  [LEAKY]" if VARIANTS[n]["leaky"] else ""
    print(f"\n{'-'*78}\n### {n}{tag}  -- {VARIANTS[n]['note']}"
          f"\n    ({len(TOKENIZER(p)['input_ids'])} tokens)\n{'-'*78}")
    print(p)
print("\n" + "=" * 78)
_tk = {n: len(TOKENIZER(render(n, 3, _demo, MAX_GUESSES, n_candidates=17))["input_ids"])
       for n in NAMES}
print("prompt length (tokens):",
      "  ".join(f"{k}={v}" for k, v in sorted(_tk.items(), key=lambda x: x[1])))
print("NOTE: longer prompts cost proportionally more decoder time.")

---
# 5. The game loop

One `play()` shared by every variant; the *only* thing that changes between
runs is which renderer `GameState.prompt` calls.

The decision cache is keyed on the prompt string, so it is automatically
per-variant and stays exact: a deterministic model plus a greedy decoder means
the same prompt always yields the same word.

In [ ]:
class GameState:
    __slots__ = ("answer","history","cands","guesses","patterns","remaining",
                 "forced","n_allowed","copied","done","solved","filt","variant")
    def __init__(self, answer, variant):
        self.answer = answer; self.variant = variant
        self.filt = HardModeFilter(LEGAL_LOWER, feedback_code)
        self.history = []; self.cands = np.arange(VOCAB.n_answers, dtype=np.int32)
        self.guesses, self.patterns, self.remaining = [], [], []
        self.forced, self.n_allowed, self.copied = [], [], []
        self.done = self.solved = False
    def prompt(self, turn):
        h = [(g.lower(), p) for g, p in self.history]
        return render(self.variant, turn, h, MAX_GUESSES,
                      n_candidates=len(self.cands))

def _shot_words(variant, history):
    """Exemplar next-guesses visible in this prompt, or () for the eleven
    variants that show none. `history` must be in the same lowercased form
    GameState.prompt passes to render(), since the rotation hashes it."""
    if variant != "few_shot":
        return ()
    h = [(g.lower(), p) for g, p in history]
    return tuple(shot_words(h))

@torch.no_grad()
def play(model, scorer, answers, variant, threshold=ADAPTIVE_THRESHOLD,
         log_every=0):
    games = [GameState(a, variant) for a in answers]
    cache, hits, seen = {}, 0, 0
    t0 = time.perf_counter()
    for turn in range(1, MAX_GUESSES + 1):
        active = [g for g in games if not g.done]
        if not active: break
        for g in active:
            n_adm = len(g.filt)
            pr = g.prompt(turn)
            seen += 1
            if USE_DECISION_CACHE and pr in cache:
                r = cache[pr]; hits += 1
            else:
                allowed = g.filt.indices(scorer) if n_adm <= threshold else None
                r = scorer.select(model, pr, allowed_idx=allowed,
                                  prune=CONSTRAINED_PRUNE)
                if USE_DECISION_CACHE: cache[pr] = r
            w, forced = r["word"], r["forced"]
            g.forced.append(forced); g.n_allowed.append(n_adm)
            g.copied.append(w.upper() in _shot_words(variant, g.history))
            c = feedback_code(w.lower(), g.answer.lower())
            g.cands = BUNDLE.fb.filter_indices(g.cands, w.lower(), c)
            g.filt.refine(w.lower(), c)
            g.history.append((w, code_to_pattern(c)))
            g.guesses.append(w); g.patterns.append(code_to_pattern(c))
            g.remaining.append(int(len(g.cands)))
            if c == ALL_GREEN: g.solved = g.done = True
            elif turn == MAX_GUESSES: g.done = True
        if log_every:
            print(f"      turn {turn}: {sum(1 for x in games if x.done)}/{len(games)}"
                  f"  {time.perf_counter()-t0:.0f}s  cache {100*hits/max(seen,1):.0f}%",
                  flush=True)
    return games, dict(secs=round(time.perf_counter()-t0, 1),
                       decisions=seen, cache_hit_pct=round(100*hits/max(seen,1), 1))

def summarize(games, label, variant):
    n = len(games)
    sc = [len(g.guesses) if g.solved else MAX_GUESSES + 1 for g in games]
    solved = [g for g in games if g.solved]
    hv = tv = 0
    for g in games:
        seen = []
        for gu, pat in zip(g.guesses, g.patterns):
            tv += 1; greens = {}
            for pg, pp in seen:
                for i, (ch, t) in enumerate(zip(pg, pp)):
                    if t == "G": greens[i] = ch
            if any(gu[i] != ch for i, ch in greens.items()): hv += 1
            seen.append((gu, pat))
    dec = [f for g in games for f in g.forced if f is not None]
    cop = [c for g in games for c in g.copied]
    return {"arm": label, "variant": variant, "leaky": VARIANTS[variant]["leaky"],
            "n_games": n, "mean": round(sum(sc)/n, 4), "solved": len(solved),
            "failure_rate_pct": round(100*(n-len(solved))/n, 2),
            "hard_mode_violation_pct": round(100*hv/max(tv, 1), 2),
            "forced_pct": round(100*sum(1 for f in dec if f)/max(len(dec), 1), 2),
            # >0 means the model is lifting exemplar words instead of reasoning;
            # few_shot is only interpretable as few-shot learning near 0.
            "shot_copy_pct": round(100*sum(cop)/max(len(cop), 1), 2),
            "opener": games[0].guesses[0] if games and games[0].guesses else None,
            "per_game": sc,
            # Per-decision |admissible|, aligned with per_guess. Phase 8 put
            # 74.7% of the classical gap at 2-10 admissible; without this the
            # only question a sweep can answer is "did the mean move", not
            # "did it move where the gap is".
            "n_allowed": [list(g.n_allowed) for g in games],
            "per_guess": [list(g.guesses) for g in games],
            "answers": [g.answer for g in games]}
print("game loop ready")

---
# 6. Resumable state

Every `(arm, variant)` result is written to disk the moment it finishes. If the
session dies at variant 9 of 24, re-running this notebook skips the first eight.

In [ ]:
SPATH = os.path.join(RESULTS_ROOT, "harness_results.json")
STATE = {"games": {}, "format": {}, "meta": {}}
if os.path.exists(SPATH):
    try:
        STATE = json.load(open(SPATH, encoding="utf-8"))
        STATE.setdefault("games", {}); STATE.setdefault("format", {})
        STATE.setdefault("meta", {})
        print(f"resuming: {len(STATE['games'])} game runs, "
              f"{len(STATE['format'])} format runs already done")
    except Exception as e:
        print("could not read previous state, starting fresh:", e)

STATE["meta"].update(n_games=len(ANSWERS), game_seed=GAME_SEED, seed=SEED,
                     threshold=ADAPTIVE_THRESHOLD, model=MODEL_NAME,
                     sft_adapter=SFT_ADAPTER, phase7=PHASE7,
                     variants=NAMES, arms=ARMS)

def save(tag=""):
    json.dump(STATE, open(SPATH, "w", encoding="utf-8"), indent=2, default=str)
    if tag: print(f"    saved [{tag}]")

MODEL_CACHE = {}
def get_model(arm):
    """One load per arm. The variants share it - reloading per variant would
    cost more than the sweep itself.

    'base' is the stock model; every other arm is an adapter name resolved
    through ARM_ADAPTERS. That indirection is what lets this notebook run the
    Phase 10 crossover (two trained adapters x two formats) without a fork."""
    if arm in MODEL_CACHE: return MODEL_CACHE[arm]
    for k in list(MODEL_CACHE):
        del MODEL_CACHE[k]
    torch.cuda.empty_cache()
    print(f"  loading arm {arm!r} ...", flush=True)
    m = base_model()
    if arm != "base":
        path = ARM_PATHS[arm]
        m = PeftModel.from_pretrained(m, path, is_trainable=False)
    m = m.to("cuda").eval()
    MODEL_CACHE[arm] = m
    return m
save()
print("state ready ->", SPATH)

---
# 7. Sanity gate

`sft` + `baseline` must reproduce Phase 7's opener and land near 3.7642. If it
does not, the adapter, the decoder, or the prompt path is wrong and every number
below is measuring something else. This has already happened once in this
project, so it is checked rather than assumed.

The gate warns rather than aborts — a 100-game subset legitimately differs from
the 246-game figure — but a *large* miss means stop.

In [ ]:
if "sft" in ARMS and "baseline" in NAMES:
    sc = build_scorer()
    m = get_model("sft")
    print("gate: sft + baseline ...", flush=True)
    gs, info = play(m, sc, ANSWERS[:40], "baseline", log_every=1)
    s = summarize(gs, "sft", "baseline")
    print(f"\n  opener {s['opener']}  (Phase 7: {PHASE7_OPENER})")
    print(f"  mean   {s['mean']}   on 40 games  (Phase 7 full-set: {PHASE7_MEAN})")
    print(f"  {info['secs']}s, {info['decisions']} decisions, "
          f"cache {info['cache_hit_pct']}%")
    ok_open = s["opener"] == PHASE7_OPENER
    ok_mean = abs(s["mean"] - PHASE7_MEAN) < 0.60
    if ok_open and ok_mean:
        print("\n  GATE PASS")
    else:
        print("\n  *** GATE FAIL ***")
        if not ok_open: print(f"      opener is {s['opener']}, expected {PHASE7_OPENER}")
        if not ok_mean: print(f"      mean {s['mean']} is far from {PHASE7_MEAN}")
        print("      Wrong adapter or wrong decoder. Fix before trusting the sweep.")
    STATE["meta"]["gate"] = {"opener": s["opener"], "mean": s["mean"],
                             "pass": bool(ok_open and ok_mean)}
    # per-decision cost, so section 8 can print a real ETA
    STATE["meta"]["secs_per_decision"] = round(info["secs"]/max(info["decisions"],1), 3)
    save("gate")
else:
    print("gate skipped (needs arm 'sft' and variant 'baseline')")

---
# 8. Probe B — format compliance, no decoder

Same variants, decoder switched off, raw greedy generation on a fixed set of
mid-game states. Three numbers per variant:

- **legal %** — is the output a real 5-letter Wordle word at all?
- **admissible %** — is it consistent with the feedback so far?
- **parse %** — did the first line even look like a word?

A prompt can raise admissible-% without changing the game mean (the decoder was
already fixing it) or raise the game mean without changing admissible-% (better
*choice* among legal words). Separating the two is the point.

**This now runs before the sweep, not after.** In the 2026-08-22 session it sat
behind the long cell, the sweep did not finish, and `format` came back empty —
so the one measurement with no lock-in confound is the one the run lost. It
costs ~2 minutes for all twelve. It goes first.

It also reports **copy %** — how often the emitted word is one of the exemplars
`few_shot` shows. That variant's 6.06 was verbatim copying, not few-shot
learning; the rate is now measured rather than noticed afterwards.

In [ ]:
@torch.no_grad()
def format_probe(model, states, variant):
    ok_parse = ok_legal = ok_adm = ok_copy = 0
    ex = []
    # per-state record so admissible-% can be split by |A| afterwards without
    # re-running: the whole point of the bands above
    per = []
    for turn, hist, adm_set, ncand in states:
        pr = render(variant, turn, hist, MAX_GUESSES, n_candidates=ncand)
        shots = _shot_words(variant, hist)
        ids = TOKENIZER(pr, return_tensors="pt").to("cuda")
        out = model.generate(**ids, max_new_tokens=FORMAT_MAX_NEW,
                             do_sample=False, num_beams=1,
                             pad_token_id=TOKENIZER.pad_token_id,
                             use_cache=True)
        txt = TOKENIZER.decode(out[0][ids["input_ids"].shape[1]:],
                               skip_special_tokens=True)
        tok = txt.strip().split("\n")[0].strip().strip('."\'*` ').upper()
        tok = "".join(ch for ch in tok if ch.isalpha())[:5]
        is_legal = is_adm = False
        if len(tok) == 5:
            ok_parse += 1
            if tok in LEGAL_GUESSES:
                ok_legal += 1; is_legal = True
                if tok.lower() in adm_set: ok_adm += 1; is_adm = True
        is_copy = bool(tok and tok in shots)
        if is_copy: ok_copy += 1
        per.append((ncand, is_legal, is_adm, is_copy))
        if len(ex) < 5: ex.append((txt.replace("\n", "\\n")[:24], tok))
    n = len(states)
    return {"variant": variant, "n": n,
            "parse_pct": round(100*ok_parse/n, 1),
            "legal_pct": round(100*ok_legal/n, 1),
            "admissible_pct": round(100*ok_adm/n, 1),
            "copy_pct": round(100*ok_copy/n, 1),
            "per_state": per,
            "examples": ex}

def _adm_set(filt):
    """HardModeFilter.words IS the surviving list - it is rebuilt in place by
    refine(), so this is a snapshot, not a view."""
    return set(filt.words)

def build_probe_states(k):
    rng = random.Random(SEED)
    # make_solver(strategy, fb, config, model) - positional. The first version
    # of this passed (BUNDLE, config) and raised on strategy.lower(). It was
    # never caught because this cell sat behind the sweep and never ran.
    solver = make_solver("entropy", BUNDLE.fb,
                         SolverConfig(strategy="entropy"), BUNDLE.model)
    # Stratified by |admissible|, not by turn. Stopping at a fixed turn against
    # an entropy solver lands almost everything at |A| = 1: the expert is too
    # good to leave interesting middles behind, which is the same effect that
    # left the training set with 58 distinct boards above |A| = 10. A probe
    # made only of forced positions cannot say where a prompt breaks, so the
    # bands are filled to a quota instead.
    bands = [(1, 1), (2, 10), (11, 50), (51, 10**9)]
    per = max(1, k // len(bands))
    got = {b: [] for b in bands}
    def band_of(n):
        for lo, hi in bands:
            if lo <= n <= hi: return (lo, hi)
        return None
    for ans in ALL_VAL:
        if all(len(v) >= per for v in got.values()): break
        filt = HardModeFilter(LEGAL_LOWER, feedback_code)
        cands = np.arange(VOCAB.n_answers, dtype=np.int32)
        hist = []
        for turn in range(1, MAX_GUESSES + 1):
            w = (solver.opening_guess() if turn == 1
                 else solver.choose(cands, turn)).lower()
            c = feedback_code(w, ans.lower())
            cands = BUNDLE.fb.filter_indices(cands, w, c)
            filt.refine(w, c)
            hist.append((w, code_to_pattern(c)))
            if c == ALL_GREEN: break
            b = band_of(len(cands))
            if b and len(got[b]) < per:
                got[b].append((turn + 1, list(hist), _adm_set(filt),
                               int(len(cands))))
                break
    out = [s for b in bands for s in got[b]]
    rng.shuffle(out)
    print("  probe bands: " + "  ".join(
        f"|A|{lo}-{'inf' if hi > 10**8 else hi}={len(got[(lo,hi)])}"
        for lo, hi in bands))
    return out[:k]

if RUN_FORMAT:
    print("building probe states ...", flush=True)
    STATES = build_probe_states(FORMAT_PROBE_STATES)
    print(f"  {len(STATES)} states, turns "
          f"{Counter(s[0] for s in STATES).most_common()}")
    for arm in ARMS:
        pending = [v for v in NAMES if f"{arm}|{v}" not in STATE["format"]]
        if not pending: continue
        model = get_model(arm)
        model.config.use_cache = True
        for v in pending:
            t0 = time.perf_counter()
            r = format_probe(model, STATES, v); r["arm"] = arm
            STATE["format"][f"{arm}|{v}"] = r
            cp = f"  copy {r['copy_pct']:>5.1f}%" if r["copy_pct"] else ""
            print(f"[{arm}] {v:<18} parse {r['parse_pct']:>5.1f}%  "
                  f"legal {r['legal_pct']:>5.1f}%  adm {r['admissible_pct']:>5.1f}%"
                  f"{cp}   {time.perf_counter()-t0:.0f}s", flush=True)
            save()
        model.config.use_cache = False
    print("\nformat probe complete")
else:
    print("RUN_FORMAT = False, skipped")

---
# 9. Probe A — the sweep

This is the long cell. It is resumable and saves after every variant, so
interrupting it loses at most one variant.

In [ ]:
sc = build_scorer()
spd = STATE["meta"].get("secs_per_decision", 0.6)
todo = [(a, v) for a in ARMS for v in NAMES if f"{a}|{v}" not in STATE["games"]]
est = len(todo) * len(ANSWERS) * 2.6 * spd / 60
print(f"{len(todo)} runs to do  |  rough ETA {est:.0f} min at {spd}s/decision\n")

if RUN_GAMES:
    for arm in ARMS:
        pending = [v for v in NAMES if f"{arm}|{v}" not in STATE["games"]]
        if not pending: continue
        model = get_model(arm)
        for i, v in enumerate(pending, 1):
            key = f"{arm}|{v}"
            t0 = time.perf_counter()
            print(f"[{arm}] {i}/{len(pending)}  {v:<18}", end="", flush=True)
            try:
                gs, info = play(model, sc, ANSWERS, v)
            except KeyboardInterrupt:
                print("  INTERRUPTED"); save("interrupt"); raise
            r = summarize(gs, arm, v); r.update(info)
            STATE["games"][key] = r
            flag = " [LEAKY]" if r["leaky"] else ""
            print(f"  mean {r['mean']:.4f}  solved {r['solved']}/{r['n_games']}"
                  f"  open {r['opener']}  {time.perf_counter()-t0:.0f}s"
                  f"  cache {info['cache_hit_pct']:.0f}%{flag}", flush=True)
            save()
    print("\nsweep complete")
else:
    print("RUN_GAMES = False, skipped")

---
# 10. Results

Paired comparison against `baseline` within each arm — same answers, same
decoder, so the only difference is the text. Paired is what makes a 0.05 gap
readable here; unpaired SE on 100 games is ~0.10 and would hide everything.

In [ ]:
def paired(a, b):
    """a, b are per-game scores on the SAME answers, in the same order."""
    d = [x - y for x, y in zip(a, b)]
    n = len(d); mu = sum(d)/n
    if n < 2: return mu, 0.0, 0, 0
    sd = math.sqrt(sum((x-mu)**2 for x in d)/(n-1))
    t = mu/(sd/math.sqrt(n)) if sd > 0 else 0.0
    return mu, t, sum(1 for x in d if x < 0), sum(1 for x in d if x > 0)

def table(arm):
    rows = [STATE["games"][k] for k in STATE["games"] if k.startswith(arm + "|")]
    if not rows: return
    base = next((r for r in rows if r["variant"] == "baseline"), None)
    rows.sort(key=lambda r: r["mean"])
    print("\n" + "=" * 92)
    print(f"ARM: {arm}    ({rows[0]['n_games']} games, adaptive@{ADAPTIVE_THRESHOLD})")
    print("=" * 92)
    print(f"{'variant':<18}{'mean':>8}{'vs base':>9}{'solved':>8}{'fail%':>7}"
          f"{'HMV%':>7}{'forced%':>9}{'better':>8}{'worse':>7}{'t':>7}  opener")
    print("-" * 92)
    for r in rows:
        if base and r["variant"] != "baseline":
            mu, t, bet, wor = paired(r["per_game"], base["per_game"])
            cmpc = f"{mu:+.4f}"; extra = f"{bet:>8}{wor:>7}{t:>7.2f}"
        else:
            cmpc = "     -"; extra = f"{'-':>8}{'-':>7}{'-':>7}"
        flag = " *LEAKY*" if r["leaky"] else ""
        print(f"{r['variant']:<18}{r['mean']:>8.4f}{cmpc:>9}"
              f"{r['solved']:>8}{r['failure_rate_pct']:>7.1f}"
              f"{r['hard_mode_violation_pct']:>7.2f}{r['forced_pct']:>9.1f}"
              f"{extra}  {r['opener']}{flag}")
    print("-" * 92)
    fair = [r for r in rows if not r["leaky"]]
    if fair:
        spread = fair[-1]["mean"] - fair[0]["mean"]
        print(f"fair spread: {spread:.4f} guesses   "
              f"(best {fair[0]['variant']} {fair[0]['mean']:.4f}, "
              f"worst {fair[-1]['variant']} {fair[-1]['mean']:.4f})")
    print(f"reference: classical entropy {PHASE7['classical_entropy']}, "
          f"filter-only {PHASE7['control_adaptive20']}, "
          f"Phase 7 full-set {PHASE7['sft_adaptive20']}")

for arm in ARMS: table(arm)

# ---- format probe -----------------------------------------------------------
for arm in ARMS:
    rows = [STATE["format"][k] for k in STATE["format"] if k.startswith(arm + "|")]
    if not rows: continue
    rows.sort(key=lambda r: -r["admissible_pct"])
    print("\n" + "=" * 72)
    print(f"FORMAT PROBE (no decoder) - arm {arm}, {rows[0]['n']} states")
    print("=" * 72)
    print(f"{'variant':<18}{'parse%':>9}{'legal%':>9}{'admissible%':>13}   sample")
    print("-" * 72)
    for r in rows:
        s = r["examples"][0][1] if r["examples"] else ""
        print(f"{r['variant']:<18}{r['parse_pct']:>9.1f}{r['legal_pct']:>9.1f}"
              f"{r['admissible_pct']:>13.1f}   {s}")

# ---- the two questions this notebook exists to answer ------------------------
print("\n" + "=" * 72)
print("VERDICT")
print("=" * 72)
for arm in ARMS:
    rows = {STATE["games"][k]["variant"]: STATE["games"][k]
            for k in STATE["games"] if k.startswith(arm + "|")}
    fair = [r for r in rows.values() if not r["leaky"]]
    if not fair: continue
    spread = max(r["mean"] for r in fair) - min(r["mean"] for r in fair)
    print(f"\n[{arm}] fair spread across variants: {spread:.4f} guesses")
    if spread < 0.05:
        print("      -> prompt format is NOT a lever on this task/model.")
    elif spread < 0.15:
        print("      -> small but real. Comparable to a full training phase.")
    else:
        print("      -> format matters MORE than four training interventions did.")
        print("         Next SFT run should use the winning format.")
    b, rh = rows.get("baseline"), rows.get("raw_history")
    if b and rh:
        mu, t, bet, wor = paired(rh["per_game"], b["per_game"])
        print(f"      raw_history vs baseline: {mu:+.4f} (t={t:.2f}, "
              f"{bet} better / {wor} worse)")
        if abs(mu) < 0.05:
            print("      -> the solver-derived constraint block is DECORATION.")
        elif mu > 0:
            print("      -> the harness has been doing the deduction, not the model.")
        else:
            print("      -> raw history BEATS the derived block; the block misleads.")
    lk = [r for r in rows.values() if r["leaky"]]
    if lk and b:
        mu, t, _, _ = paired(lk[0]["per_game"], b["per_game"])
        print(f"      [leaky] candidate count is worth {(-mu):+.4f} - an upper "
              f"bound only, not usable.")
save("final")

---
# 11. Save

Publishes to a Kaggle Dataset if the secrets are set, and always writes a zip so
there is a download either way.

In [ ]:
import zipfile
SLUG  = "wordle-phase9-harness"
TITLE = "Wordle Phase 9 - prompt/harness sweep"

save("pre-zip")
json.dump({k: {kk: vv for kk, vv in v.items() if kk != "per_game"}
           for k, v in STATE["games"].items()},
          open(os.path.join(RESULTS_ROOT, "summary.json"), "w"), indent=2)

if os.path.exists(RESULTS_ZIP): os.remove(RESULTS_ZIP)
with zipfile.ZipFile(RESULTS_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(RESULTS_ROOT):
        for f in files:
            p = os.path.join(root, f)
            z.write(p, os.path.relpath(p, RESULTS_ROOT))
print(f"zip: {RESULTS_ZIP} ({os.path.getsize(RESULTS_ZIP)/2**20:.2f} MiB)")
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(RESULTS_ZIP, "/kaggle/working")))
except Exception: pass

# Auto-publish is a CONVENIENCE. It must never decide whether the run passed.
#
# The 2026-08-23 session completed every measurement, saved, and zipped - then
# this block raised SystemExit because no Kaggle secrets are configured, and
# IPython crashed formatting that exception ('tuple' object has no attribute
# 'f_lineno'). Kaggle reported the whole run as ERROR. The results were sitting
# in the zip the entire time. Nothing here is allowed to raise.
USER = None
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _s.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _s.get_secret("KAGGLE_KEY")
    USER = os.environ["KAGGLE_USERNAME"]
except Exception as e:
    print(f"no Kaggle credentials ({type(e).__name__}) - results are in the zip "
          f"above and in {RESULTS_ROOT}. This is not a failure.")

if USER:
    try:
        json.dump({"title": TITLE, "id": f"{USER}/{SLUG}",
                   "licenses": [{"name": "CC0-1.0"}]},
                  open(os.path.join(RESULTS_ROOT, "dataset-metadata.json"), "w"),
                  indent=2)
        def run(c):
            r = subprocess.run(c, capture_output=True, text=True)
            return r.returncode, (r.stdout or "") + (r.stderr or "")
        rc, out = run(["kaggle", "datasets", "version", "-p", RESULTS_ROOT,
                       "-m", f"phase9 {time.strftime('%Y-%m-%d %H:%M')}", "-r", "zip"])
        if rc != 0 and ("404" in out or "not found" in out.lower()):
            rc, out = run(["kaggle", "datasets", "create", "-p", RESULTS_ROOT,
                           "-r", "zip"])
        print(f"PUBLISHED -> https://www.kaggle.com/datasets/{USER}/{SLUG}" if rc == 0
              else "publish failed - use the zip\n" + out.strip()[:600])
    except Exception as e:
        print(f"publish failed ({type(e).__name__}: {e}) - use the zip")

print("\nRUN COMPLETE - all measurements saved.")